# Requirement 3 - Best of Both Worlds

This notebook is the Requirement 3 deliverable for the `giovanni` branch. It documents and reproduces the Primal-Dual multi-campaign experiment implemented in `utils.run_req3`.

The same agent is evaluated in two regimes:

- a stochastic multi-campaign environment, compared against the fixed LP optimum;
- an adversarial/non-stationary environment, compared against `OPT^A`, the best fixed distribution in hindsight.

The second benchmark is important: the best-of-both-worlds claim is about competing with a fixed distribution in hindsight, not with a dynamic per-round clairvoyant.


In [ ]:
from pathlib import Path
import pickle
import pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'picklefiles'
OUT = ROOT / 'outputs' / 'r3'
BUDGET = 1600.0


## Reproduce the experiment

Run this command from the repository root to regenerate the pickles and plots used below.


In [ ]:
# !MPLBACKEND=Agg uv run python -m utils.run_req3

# Equivalent Python entry point:
# from utils.run_req3 import run_req3
# run_req3()


## Numerical summary

The generated results use `T = 10000`, `B = 1600`, `20` trials, `env_mode = drift`, `budget_pacing = True`, and `ogd_eta = 0.017`, matching the current `giovanni` branch configuration.


In [ ]:
def load_result(name):
    with open(DATA / f'{name}_results.pkl', 'rb') as f:
        return pickle.load(f)

results = {
    'Primal-Dual stochastic': load_result('req3_stochastic'),
    'Primal-Dual adversarial': load_result('req3_adversarial'),
}

rows = []
for label, res in results.items():
    rows.append({
        'setting': label,
        'final_regret': float(res['mean_regret'][-1]),
        'final_cost': float(res['mean_cumcost'][-1]),
        'unused_budget': float(BUDGET - res['mean_cumcost'][-1]),
        'final_lambda': float(res['mean_lmbd'][-1]),
    })
summary = pd.DataFrame(rows)
display(summary)


## Main plots

The annotated regret plot is the main slide candidate. The budget and lambda plots explain feasibility and dual-price dynamics.


In [ ]:
for name in [
    'req3_regret_annotated.png',
    'req3_regret.png',
    'req3_budget.png',
    'req3_lambda.png',
]:
    display(Image(filename=str(OUT / name)))


## Interpretation notes

- The primal side uses Hedge over feasible campaign/bid decisions; the dual side updates a shared Lagrange multiplier for the budget constraint.
- In the stochastic regime, the benchmark is the LP optimum computed from true win probabilities.
- In the adversarial regime, the benchmark is `OPT^A`, the best fixed distribution in hindsight computed from empirical win probabilities of the realised sequence.
- A dynamic per-round clairvoyant is not the right Requirement 3 benchmark: in a strongly non-stationary sequence it would make regret linear by construction.
- The `giovanni` branch currently runs the agent with budget pacing enabled. This improves budget feasibility in finite horizon experiments, but when discussing theory it should be identified as the branch configuration rather than silently conflated with the minimal Hedge+OGD template.
- Lambda is a shadow price for budget pressure. The adversarial run can have a higher final lambda because the realised sequence creates stronger pressure on feasible spending.
